In [2]:
import os
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from dotenv import load_dotenv

car_code = pd.read_excel("4054_차명코드(251223).xlsx")
car_code["cnmCode"]=car_code["코드"].astype(int).astype(str).str.zfill(6)


In [ ]:
# 아반떼 차명 모음 조회
df = pd.read_excel("4054_차명코드(251223).xlsx")
df["cnmCode"] = df["코드"].astype(int).astype(str).str.zfill(6)

keyword = "아반떼"
cand = df[df["코드명"].str.contains(keyword, na=False)][["cnmCode","코드명"]]
print(cand.sort_values("코드명").head(50))  # 후보 확인


In [5]:

def call_api(url, params):
    r = requests.get(url, params=params, timeout=30)
    txt = (r.text or "").strip()
    
    # XML 파싱
    try:
        root = ET.fromstring(txt)
    except ET.ParseError:
        return {"ok": False, "status": r.status_code, "text_head": txt[:300], "url": r.url}

    # 정상/에러 코드 읽기 (문서 예시 구조)  :contentReference[oaicite:4]{index=4}
    result_code = root.findtext("./header/resultCode")
    result_msg  = root.findtext("./header/resultMsg")
    dtaCo       = root.findtext("./body/dtaCo")

    return {"ok": True, "resultCode": result_code, "resultMsg": result_msg, "dtaCo": dtaCo, "url": r.url}

def crawling_car():
    load_dotenv()
    key = os.getenv("KEY")

    url = "https://apis.data.go.kr/B553881/newRegistlnfoService_02/getnewRegistlnfoService02"  # 문서 서비스 URL :contentReference[oaicite:5]{index=5}

    fuel_codes = ["2", "5", "7", "8"]
    sexes = ["남자", "여자"]
    months = [f"{m:02d}" for m in range(1, 3)]

    rows = []
    bad = []

    for agecode in range(1, 9):         # 10대~80대
        for mt in months:               # ✅ registMt 필수
            for fuel in fuel_codes:
                for sex in sexes:
                    params = {
                        "serviceKey": key,      
                        "registYy": "2025",     # 필수 :contentReference[oaicite:7]{index=7}
                        "registMt": mt,         # ✅ 필수 :contentReference[oaicite:8]{index=8}
                        "vhctyAsortCode": "1",
                        "registGrcCode": "1",
                        "useFuelCode": fuel,
                        "prposSeNm": "1",
                        "sexdstn": sex,
                        "agrde": str(agecode),
                        "prye": 2025,
                    }

                    res = call_api(url, params)

                    if not res["ok"] or res.get("resultCode") not in (None, "00"):
                        bad.append({
                            "agecode": agecode, "registMt": mt, "fuel": fuel, "sex": sex, "prye": 2025,
                            "status": res.get("status"), "resultCode": res.get("resultCode"),
                            "resultMsg": res.get("resultMsg"), "text_head": res.get("text_head"),
                        })
                        continue

                    rows.append({
                        "registYy": 2025, "registMt": mt, "agrde": agecode,
                        "useFuelCode": fuel, "sexdstn": sex, "prye": 2025,
                        "dtaCo": int(res["dtaCo"]) if res["dtaCo"] and res["dtaCo"].isdigit() else None

                    })

    df = pd.DataFrame(rows)
    bad_df = pd.DataFrame(bad)
    return df, bad_df

if __name__ == "__main__":
    df, bad_df = crawling_car()
    print(df.head())
    print("OK rows:", len(df))
    print("BAD rows:", len(bad_df))
    if len(bad_df):
        print(bad_df.head(10))


   registYy registMt  agrde useFuelCode sexdstn  prye  dtaCo
0      2025       01      1           2      남자  2025      4
1      2025       01      1           7      남자  2025      9
2      2025       01      1           7      여자  2025      3
3      2025       01      1           8      여자  2025      1
4      2025       02      1           2      남자  2025      2
OK rows: 108
BAD rows: 20
   agecode registMt fuel sex  prye status resultCode     resultMsg text_head
0        1       01    2  여자  2025   None         03  NODATA_ERROR      None
1        1       01    5  남자  2025   None         03  NODATA_ERROR      None
2        1       01    5  여자  2025   None         03  NODATA_ERROR      None
3        1       01    8  남자  2025   None         03  NODATA_ERROR      None
4        1       02    2  여자  2025   None         03  NODATA_ERROR      None
5        1       02    5  남자  2025   None         03  NODATA_ERROR      None
6        1       02    5  여자  2025   None         03  NODATA_ERROR   

In [31]:
df.to_csv("stocks.csv", index=False, encoding="cp949")

In [1]:
"""
신규등록정보(OpenAPI) 크롤러: 연령×성별×분기 단위로
- 전체(분모) 신규등록 수
- (차명그룹 cnmCode 묶음) × (유종 useFuelCode) 신규등록 수
- 비율(그룹/전체) 계산

✅ 핵심 포인트
- API는 registYy + registMt(월) 단위라서, 분기는 3개월 합산으로 만듭니다.
- cnmCode, useFuelCode는 "한 번에 하나 값"만 넣고 호출하는 것을 권장합니다.
- cnmCode 그룹은 여러 코드 합산으로 처리합니다.

필요사항:
pip install requests pandas python-dotenv openpyxl
.env 파일에 KEY=발급받은 서비스키(Decoding 키 권장)
"""

import os
import time
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Dict, List, Optional, Tuple

# -----------------------------
# 1) 환경 설정
# -----------------------------

# ✅ 사용 중인 엔드포인트를 여기에 넣으세요.
# 사용자 환경에서 브라우저로 성공한 URL의 "?serviceKey=..." 앞부분을 그대로 쓰는 게 가장 안전합니다.
BASE_URL = "https://apis.data.go.kr/B553881/newRegistInfoService_02/getnewRegistInfoService02"
# 만약 다른 엔드포인트가 성공했다면 위를 교체하세요.
# 예) "https://apis.data.go.kr/B553881/newRegistlnfoService_02/getnewRegistlnfoService02" (철자 차이 가능)

YEAR = "2025"  # 분석 연도 (문자열로 권장)

# 성별: 문서 기준 "남자", "여자", "법인"
SEXES = ["남자", "여자"]

# 연령대: 1=10대 ... 8=80대
AGES = [str(i) for i in range(1, 9)]

# 유종 코드 (필요한 것만 사용)
# 2:경유, 5:전기, 7:하이브리드(휘발유+전기), 8:휘발유
FUEL_CODES = {
    "DIESEL": "2",
    "EV": "5",
    "HEV_GAS": "7",
    "GAS": "8",
    # 필요하면 추가:
    # "LPG": "4",
    # "HYB_CNG": "6",
    # "H2": "3",
}

# 분기 → 월 목록
QUARTERS = {
    "Q1": ["01", "02", "03"],
    "Q2": ["04", "05", "06"],
    "Q3": ["07", "08", "09"],
    "Q4": ["10", "11", "12"],
}

# (선택) 추가 조건을 고정으로 걸고 싶으면 여기에 넣으세요.
# 예: 승용만, 서울만, 자가용만 등
FIXED_PARAMS = {
    # "vhctyAsortCode": "1",   # 승용
    # "registGrcCode": "1",    # 서울
    # "prposSeNm": "1",        # 자가용
}

# ✅ 대표 차명 그룹 (cnmCode 묶음)
# - 유종 분리 + N/N Line 분리를 하고 싶다면 그룹 자체를 그렇게 나누는 방식이 가장 관리가 쉽습니다.
# - 아래는 예시(우리가 앞에서 만든 추천 매핑 일부)입니다. 필요 모델을 추가/수정하세요.
CAR_GROUPS: Dict[str, List[str]] = {
    # 아반떼
    "AVANTE_ICE_BASE": ["003869"],  # 아반떼(대표)
    # 아반떼 파생은 코드표에서 더 골라서 아래처럼 추가 가능:
    # "AVANTE_HEV": ["040636"],  # 아반떼 Hybrid
    # "AVANTE_N": ["048602"],    # 아반떼 N
    # "AVANTE_NLINE": ["048603", "061106", "040637", "040638"],

    # 그랜저
    "GRANDEUR_ICE_BASE": ["001236"],
    #K5
    "K5_ICE_BASE":["044682"],
    #K9
    "K9_ICE_BASE":["058121"],
    # G80
    "G80_ICE_BASE":["008243"],

    # SUV 예시
    "SELTOS_ICE_BASE": ["039852"],  # 셀토스
    "KONA_ICE_BASE": ["006987"],  # 코나(KONA)
    "SORENTO_ICE_BASE": ["025012"], # 쏘렌토
    "SANTAFE_ICE_BASE":["039388"], # 산타페
    "TUCSAN_ICE_BASE":["024028"], # 투싼

    # EV 전용 차명(차명 자체가 EV인 경우가 많아 유종은 EV(5)와 같이 쓰는 걸 추천)
    "IONIQ5_EV": ["040681"],
    "EV6_EV": ["030030"],
    "TESLA_MODEL3_EV": ["047007"],
}

# 병렬 워커 수(너무 높이면 500/차단 가능)
MAX_WORKERS = 10

# API 호출 간 딜레이(서버 상태에 따라 조정)
SLEEP_BETWEEN_REQUESTS = 0.0


# -----------------------------
# 2) API 유틸
# -----------------------------

def _safe_int(x: Optional[str]) -> int:
    if not x:
        return 0
    x = x.strip()
    return int(x) if x.isdigit() else 0


def fetch_dtaCo(
    session: requests.Session,
    base_url: str,
    service_key: str,
    registYy: str,
    registMt: str,
    extra_params: Dict[str, str],
) -> Tuple[int, Optional[str]]:
    """
    단일 요청에서 dtaCo를 반환.
    실패 시 (0, 에러설명) 형태로 반환.
    """
    params = {
        "serviceKey": service_key,
        "registYy": registYy,
        "registMt": registMt,
        **extra_params,
    }

    try:
        r = session.get(base_url, params=params, timeout=25, headers={"User-Agent": "Mozilla/5.0"})
        txt = (r.text or "").strip()

        if SLEEP_BETWEEN_REQUESTS > 0:
            time.sleep(SLEEP_BETWEEN_REQUESTS)

        # XML이 아니거나 200이 아니면 실패
        if r.status_code != 200 or not txt.startswith("<"):
            return 0, f"HTTP {r.status_code}: {txt[:120]}"

        root = ET.fromstring(txt)

        # 결과 코드 체크 (있을 수도/없을 수도)
        result_code = root.findtext(".//resultCode")
        result_msg = root.findtext(".//resultMsg")
        if result_code and result_code != "00":
            return 0, f"API {result_code}: {result_msg}"

        dta = root.findtext(".//dtaCo")
        return _safe_int(dta), None

    except ET.ParseError as e:
        return 0, f"XML ParseError: {str(e)}"
    except requests.RequestException as e:
        return 0, f"RequestException: {str(e)}"
    except Exception as e:
        return 0, f"UnknownError: {str(e)}"


def fetch_quarter_sum(
    session: requests.Session,
    base_url: str,
    service_key: str,
    year: str,
    quarter: str,
    extra_params: Dict[str, str],
) -> Tuple[int, List[str]]:
    """
    분기(3개월) 합산 dtaCo를 반환.
    """
    months = QUARTERS[quarter]
    total = 0
    errors: List[str] = []
    for mt in months:
        v, err = fetch_dtaCo(session, base_url, service_key, year, mt, extra_params)
        total += v
        if err:
            errors.append(f"{year}-{mt} {err}")
    return total, errors


# -----------------------------
# 3) 크롤링 로직
# -----------------------------

def make_jobs() -> List[dict]:
    """
    (분모) 전체: cnmCode/useFuelCode 없이
    (분자) 그룹×유종: cnmCode 1개씩 + useFuelCode 1개
    """
    jobs = []

    # 3-1) 분모(전체): 연령×성별×분기
    for sex in SEXES:
        for age in AGES:
            for q in QUARTERS.keys():
                jobs.append({
                    "job_type": "BASELINE",
                    "sex": sex,
                    "age": age,
                    "quarter": q,
                    "group": "ALL",
                    "fuel_key": "ALL",
                    "cnmCode": None,
                    "useFuelCode": None,
                })

    # 3-2) 분자(그룹×유종): 연령×성별×분기×그룹×유종×(그룹 내 cnmCode들)
    for sex in SEXES:
        for age in AGES:
            for q in QUARTERS.keys():
                for group_name, cnm_list in CAR_GROUPS.items():
                    for fuel_key, fuel_code in FUEL_CODES.items():
                        for cnm in cnm_list:
                            jobs.append({
                                "job_type": "NUMERATOR",
                                "sex": sex,
                                "age": age,
                                "quarter": q,
                                "group": group_name,
                                "fuel_key": fuel_key,
                                "cnmCode": cnm,
                                "useFuelCode": fuel_code,
                            })

    return jobs


def run_crawl() -> Tuple[pd.DataFrame, pd.DataFrame]:
    load_dotenv()
    key = os.getenv("KEY")
    if not key:
        raise RuntimeError("환경변수 KEY가 없습니다. .env에 KEY=... 를 설정하세요.")

    session = requests.Session()

    jobs = make_jobs()
    results = []
    errors = []

    def worker(job: dict):
        # 요청 파라미터 구성
        extra = dict(FIXED_PARAMS)  # 복사

        # 공통 필터: 성별/연령
        extra["sexdstn"] = job["sex"]
        extra["agrde"] = job["age"]

        if job["job_type"] == "NUMERATOR":
            extra["cnmCode"] = job["cnmCode"]
            extra["useFuelCode"] = job["useFuelCode"]
        # BASELINE은 cnmCode/useFuelCode 없음

        total, errs = fetch_quarter_sum(
            session=session,
            base_url=BASE_URL,
            service_key=key,
            year=YEAR,
            quarter=job["quarter"],
            extra_params=extra
        )

        return job, total, errs

    # 병렬 실행
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        future_map = {ex.submit(worker, j): j for j in jobs}
        for fut in as_completed(future_map):
            job, total, errs = fut.result()

            results.append({
                "year": YEAR,
                "quarter": job["quarter"],
                "sex": job["sex"],
                "age": job["age"],
                "job_type": job["job_type"],
                "group": job["group"],
                "fuel": job["fuel_key"],
                "cnmCode": job["cnmCode"],
                "useFuelCode": job["useFuelCode"],
                "count": total,
            })

            for e in errs:
                errors.append({
                    "year": YEAR,
                    "quarter": job["quarter"],
                    "sex": job["sex"],
                    "age": job["age"],
                    "job_type": job["job_type"],
                    "group": job["group"],
                    "fuel": job["fuel_key"],
                    "cnmCode": job["cnmCode"],
                    "useFuelCode": job["useFuelCode"],
                    "error": e,
                })

    df = pd.DataFrame(results)
    err_df = pd.DataFrame(errors)

    # -----------------------------
    # 4) 후처리: 그룹+유종 합산, 비율 계산
    # -----------------------------

    # 분모
    base = df[df["job_type"] == "BASELINE"][[
        "year", "quarter", "sex", "age", "count"
    ]].rename(columns={"count": "base_total"})

    # 분자: cnmCode별 결과를 group+fuel로 합산
    num = df[df["job_type"] == "NUMERATOR"].groupby(
        ["year", "quarter", "sex", "age", "group", "fuel"],
        as_index=False
    )["count"].sum().rename(columns={"count": "group_fuel_total"})

    # 병합 후 비율
    merged = num.merge(base, on=["year", "quarter", "sex", "age"], how="left")
    merged["share"] = merged.apply(
        lambda r: (r["group_fuel_total"] / r["base_total"]) if r["base_total"] and r["base_total"] > 0 else 0.0,
        axis=1
    )

    # 보기 좋게 정렬
    merged = merged.sort_values(
        ["year", "quarter", "sex", "age", "share"],
        ascending=[True, True, True, True, False]
    )

    return merged, err_df


def save_outputs(result_df: pd.DataFrame, err_df: pd.DataFrame):
    out_dir = os.getcwd()
    out_csv = os.path.join(out_dir, f"new_regist_by_quarter_{YEAR}.csv")
    out_err = os.path.join(out_dir, f"new_regist_errors_{YEAR}.csv")
    out_xlsx = os.path.join(out_dir, f"new_regist_by_quarter_{YEAR}.xlsx")

    result_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    err_df.to_csv(out_err, index=False, encoding="utf-8-sig")

    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        result_df.to_excel(writer, index=False, sheet_name="result")
        err_df.to_excel(writer, index=False, sheet_name="errors")

    print("Saved:")
    print(out_csv)
    print(out_err)
    print(out_xlsx)


if __name__ == "__main__":
    result_df, err_df = run_crawl()
    print(result_df.head(30))
    print("result rows:", len(result_df))
    print("error rows:", len(err_df))
    save_outputs(result_df, err_df)


    year quarter sex age              group     fuel  group_fuel_total  \
0   2025      Q1  남자   1    AVANTE_ICE_BASE   DIESEL                 0   
1   2025      Q1  남자   1    AVANTE_ICE_BASE       EV                 0   
2   2025      Q1  남자   1    AVANTE_ICE_BASE      GAS                 0   
3   2025      Q1  남자   1    AVANTE_ICE_BASE  HEV_GAS                 0   
4   2025      Q1  남자   1             EV6_EV   DIESEL                 0   
5   2025      Q1  남자   1             EV6_EV       EV                 0   
6   2025      Q1  남자   1             EV6_EV      GAS                 0   
7   2025      Q1  남자   1             EV6_EV  HEV_GAS                 0   
8   2025      Q1  남자   1       G80_ICE_BASE   DIESEL                 0   
9   2025      Q1  남자   1       G80_ICE_BASE       EV                 0   
10  2025      Q1  남자   1       G80_ICE_BASE      GAS                 0   
11  2025      Q1  남자   1       G80_ICE_BASE  HEV_GAS                 0   
12  2025      Q1  남자   1  GRANDEUR_ICE

In [4]:
"""
아반떼(차명코드)만 뽑아서 신규등록(dtaCo) 크롤링하는 최소 실행 코드

✅ 하는 일
1) 차명코드표(엑셀)에서 '아반떼' 후보 cnmCode를 찾음
2) API로 2025-01 기준 >0 나오는 cnmCode만 "살아있는 코드"로 검증
3) 선택된 cnmCode(1개 또는 여러개)를 이용해
   - 분기(Q1~Q4) 단위(월 3개 합산)
   - 연령(1~8) × 성별(남/여)
   - 유종(useFuelCode)별(휘발유/경유/HEV/EV)
   결과를 DataFrame으로 생성

필요:
pip install requests pandas python-dotenv openpyxl
.env에 KEY=서비스키(Decoding 키 권장)
"""

import os
import re
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from dotenv import load_dotenv
from typing import Dict, List, Tuple, Optional

# ✅ 브라우저에서 "값이 나오는" 엔드포인트(물음표 앞까지)를 그대로 넣으세요.
BASE_URL = "https://apis.data.go.kr/B553881/newRegistInfoService_02/getnewRegistInfoService02"

YEAR = "2025"

QUARTERS = {
    "Q1": ["01", "02", "03"],
    "Q2": ["04", "05", "06"],
    "Q3": ["07", "08", "09"],
    "Q4": ["10", "11", "12"],
}

SEXES = ["남자", "여자"]
AGES = [str(i) for i in range(1, 9)]  # 1=10대 ... 8=80대

# # 유종 코드(원하는 것만)
FUEL_CODES = {
    "GAS": "8",       # 휘발유
    "DIESEL": "2",    # 경유
    "HEV": "7",       # 하이브리드(휘발유+전기)
    "EV": "5",        # 전기
}

# (선택) 고정조건(승용/지역/자가용 등) 원하면 켜세요.
FIXED_PARAMS = {
    # "vhctyAsortCode": "1",
    # "registGrcCode": "1",
    # "prposSeNm": "1",
}

# ✅ 차명코드표 파일 경로(업로드한 파일 기준)
CODEBOOK_XLSX = r"4054_차명코드(251223).xlsx"


# ---------------------------
# API 호출 (dtaCo)
# ---------------------------
def _safe_int(x: Optional[str]) -> int:
    if not x:
        return 0
    x = x.strip()
    return int(x) if x.isdigit() else 0


def fetch_dtaCo(session: requests.Session, key: str, yy: str, mt: str, extra: Dict[str, str]) -> Tuple[int, str]:
    params = {
        "serviceKey": key,
        "registYy": yy,
        "registMt": mt,
        **extra,
    }
    r = session.get(BASE_URL, params=params, timeout=25, headers={"User-Agent": "Mozilla/5.0"})
    txt = (r.text or "").strip()

    # XML이 아니거나 200 아니면 에러로 처리
    if r.status_code != 200 or not txt.startswith("<"):
        return 0, f"HTTP {r.status_code}: {txt[:120]}"

    try:
        root = ET.fromstring(txt)
    except ET.ParseError as e:
        return 0, f"XML ParseError: {str(e)}"

    result_code = root.findtext(".//resultCode")
    result_msg = root.findtext(".//resultMsg")
    if result_code and result_code != "00":
        return 0, f"API {result_code}: {result_msg}"

    dta = root.findtext(".//dtaCo")
    return _safe_int(dta), "OK"


def fetch_quarter_sum(session: requests.Session, key: str, yy: str, q: str, extra: Dict[str, str]) -> Tuple[int, List[str]]:
    total = 0
    errs = []
    for mt in QUARTERS[q]:
        v, msg = fetch_dtaCo(session, key, yy, mt, extra)
        total += v
        if msg != "OK":
            errs.append(f"{yy}-{mt} {msg}")
    return total, errs


# ---------------------------
# 1) 차명코드표에서 '아반떼' 후보 찾기
# ---------------------------
def load_codebook() -> pd.DataFrame:
    df = pd.read_excel(CODEBOOK_XLSX)
    df["cnmCode"] = df["코드"].astype(int).astype(str).str.zfill(6)
    return df[["cnmCode", "코드명"]].copy()


def find_avante_candidates(df: pd.DataFrame, topn: int = 50) -> pd.DataFrame:
    # '아반떼' 포함 후보
    cand = df[df["코드명"].str.contains("아반떼", na=False)].copy()
    # 너무 넓으면 상위 n개만(가나다 순)
    cand = cand.sort_values("코드명").head(topn)
    return cand


# ---------------------------
# 2) 후보 cnmCode 중 "살아있는 코드" 자동 검증
# ---------------------------
def validate_alive_cnm_codes(session: requests.Session, key: str, cand: pd.DataFrame, yy="2025", mt="01") -> pd.DataFrame:
    alive_rows = []
    for _, row in cand.iterrows():
        cnm = row["cnmCode"]
        v, msg = fetch_dtaCo(session, key, yy, mt, {"cnmCode": cnm, **FIXED_PARAMS})
        # msg가 OK인데 0이면 "그 달에는 신규등록이 없을 수도" 있으니,
        # 확실히 하려면 2~3개월 더 체크할 수도 있음(여기서는 1개월로 간단히)
        if msg == "OK" and v > 0:
            alive_rows.append({"cnmCode": cnm, "코드명": row["코드명"], "count_2025_01": v})

    out = pd.DataFrame(alive_rows, columns=["cnmCode", "코드명", "count_2025_01"])
    if out.empty:
        return out
    return out.sort_values("count_2025_01", ascending=False)


# ---------------------------
# 3) 아반떼만 크롤링(분기×연령×성별×유종)
# ---------------------------
def crawl_avante_only(alive_cnm_codes: List[str]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    load_dotenv()
    key = os.getenv("KEY")
    if not key:
        raise RuntimeError(".env에 KEY=... 를 설정하세요.")

    session = requests.Session()

    rows = []
    errs = []

    for sex in SEXES:
        for age in AGES:
            for q in QUARTERS.keys():
                for fuel_name, fuel_code in FUEL_CODES.items():
                    # 아반떼 파생(아반떼/아반떼 하이브리드/아반떼 N/N Line 등)을
                    # 여러 cnmCode로 묶고 싶으면 여기서 합산
                    subtotal = 0
                    all_errs = []

                    for cnm in alive_cnm_codes:
                        extra = {
                            **FIXED_PARAMS,
                            "sexdstn": sex,
                            "agrde": age,
                            "cnmCode": cnm,
                            "useFuelCode": fuel_code,
                        }
                        v, e = fetch_quarter_sum(session, key, YEAR, q, extra)
                        subtotal += v
                        all_errs.extend(e)

                    rows.append({
                        "year": YEAR,
                        "quarter": q,
                        "sex": sex,
                        "age": age,
                        "fuel": fuel_name,
                        "count": subtotal,
                        "cnmCode_used": ",".join(alive_cnm_codes),
                    })

                    for e in all_errs:
                        errs.append({
                            "year": YEAR,
                            "quarter": q,
                            "sex": sex,
                            "age": age,
                            "fuel": fuel_name,
                            "cnmCode_used": ",".join(alive_cnm_codes),
                            "error": e,
                        })

    return pd.DataFrame(rows), pd.DataFrame(errs)


if __name__ == "__main__":
    load_dotenv()
    key = os.getenv("KEY")
    if not key:
        raise RuntimeError(".env에 KEY=... 를 설정하세요.")

    session = requests.Session()

    # 1) 코드표 로드 + 아반떼 후보 찾기
    codebook = load_codebook()
    avante_cand = find_avante_candidates(codebook, topn=80)
    print(f"[1] 아반떼 후보 개수: {len(avante_cand)}")

    # 2) 2025-01 기준으로 살아있는 cnmCode 검증
    alive_df = validate_alive_cnm_codes(session, key, avante_cand, yy="2025", mt="01")
    print(f"[2] 2025-01 기준 살아있는 cnmCode 개수: {len(alive_df)}")
    print(alive_df.head(10))

    if alive_df.empty:
        raise RuntimeError(
            "아반떼 후보가 모두 0입니다. (1) 다른 월로 검증해보거나 (2) 엔드포인트/키 확인 필요"
        )

    # ✅ 여기서 선택 전략:
    # - 가장 많이 잡히는 1개만 쓰기: top1
    # - 파생까지 묶고 싶으면 상위 여러개(topK) 사용
    top1 = [alive_df.iloc[0]["cnmCode"]]
    # topK 예: [..][:3]
    # topK = alive_df["cnmCode"].head(3).tolist()

    print(f"[3] 사용할 아반떼 cnmCode: {top1}")

    # 3) 아반떼만 크롤링
    result_df, err_df = crawl_avante_only(top1)

    print("[DONE] result head:")
    print(result_df.head(20))
    print("result rows:", len(result_df))
    print("error rows:", len(err_df))

    # 저장
    out_result = f"avante_only_{YEAR}_by_quarter.csv"
    out_err = f"avante_only_{YEAR}_errors.csv"
    result_df.to_csv(out_result, index=False, encoding="utf-8-sig")
    err_df.to_csv(out_err, index=False, encoding="utf-8-sig")
    print("Saved:", out_result, out_err)


[1] 아반떼 후보 개수: 57
[2] 2025-01 기준 살아있는 cnmCode 개수: 0
Empty DataFrame
Columns: [cnmCode, 코드명, count_2025_01]
Index: []


RuntimeError: 아반떼 후보가 모두 0입니다. (1) 다른 월로 검증해보거나 (2) 엔드포인트/키 확인 필요

In [7]:
from urllib.parse import urlencode
import requests

key = os.getenv("KEY")
base = "https://apis.data.go.kr/B553881/newRegistInfoService_02/getnewRegistInfoService02"
query = {
    "serviceKey": key, "registYy":"2025", "registMt":"01",
    # "cnmCode":"003869"
}
url = base + "?" + urlencode(query, doseq=False)
r = requests.get(url, timeout=20, headers={"User-Agent":"Mozilla/5.0"})
print(r.status_code, r.text[:200])


500 Unexpected errors



In [11]:
import os, time, requests, xml.etree.ElementTree as ET
from dotenv import load_dotenv

load_dotenv()
KEY = os.getenv("KEY")  # .env에 KEY=... (너의 키)
BASE_URL = "https://apis.data.go.kr/B553881/newRegistlnfoService_02/getnewRegistlnfoService02"

def call_new_regist(
    registYy="2025",
    registMt="12",
    vhctyAsortCode="1",
    sexdstn="남자",
    agrde="3",
    hmmdImpSeNm="국산",
    cnmCode=None,
    useFuelCode=None,
    retries=3,
):
    params = {
        "serviceKey": KEY,                 # 인증키 (URL-Encode라고 문서에 적혀있지만, params 방식이면 보통 디코딩 키가 안정적) :contentReference[oaicite:2]{index=2}
        "registYy": str(registYy),
        "registMt": f"{int(registMt):02d}",
        "vhctyAsortCode": str(vhctyAsortCode),  # 1:승용 :contentReference[oaicite:3]{index=3}
        "sexdstn": sexdstn,                # 남자/여자/법인 :contentReference[oaicite:4]{index=4}
        "cnmCode": "001236",
        "agrde": str(agrde),               # 1~8 (10대~80대) :contentReference[oaicite:5]{index=5}
        "hmmdImpSeNm": hmmdImpSeNm,        # 국산/외산 :contentReference[oaicite:6]{index=6}
    }

    # ✅ 옵션은 "값이 있을 때만" 넣기 (None/리스트 넣지 말기)
    if cnmCode:
        params["cnmCode"] = str(cnmCode)   # 차명코드(코드표 참조) :contentReference[oaicite:7]{index=7}
    if useFuelCode:
        params["useFuelCode"] = str(useFuelCode)  # 유종코드 :contentReference[oaicite:8]{index=8}

    last_err = None
    for i in range(retries):
        r = requests.get(BASE_URL, params=params, timeout=20, headers={"User-Agent": "Mozilla/5.0"})
        txt = (r.text or "").strip()

        # 500이면 잠깐 쉬고 재시도
        if r.status_code >= 500:
            last_err = f"HTTP {r.status_code}: {txt[:120]}"
            time.sleep(0.7 * (2 ** i))
            continue

        # 200인데 XML이 아니면 실패
        if r.status_code != 200 or not txt.startswith("<"):
            raise RuntimeError(f"BAD {r.status_code} {txt[:200]}")

        root = ET.fromstring(txt)
        resultCode = root.findtext(".//resultCode")
        resultMsg = root.findtext(".//resultMsg")
        dtaCo = root.findtext(".//dtaCo")  # 통계 건수 :contentReference[oaicite:9]{index=9}

        return {
            "url": r.url,
            "resultCode": resultCode,
            "resultMsg": resultMsg,
            "dtaCo": int(dtaCo) if dtaCo and dtaCo.isdigit() else 0,
        }

    raise RuntimeError(f"Server keeps failing: {last_err}")

# ✅ 네가 보여준 케이스 그대로(차명코드 없이)
print(call_new_regist())

# ✅ 차명코드/유종까지 붙여서 테스트(예: 아반떼 cnmCode 넣고 싶으면 아래처럼)
# print(call_new_regist(cnmCode="003869", useFuelCode="8"))


{'url': 'https://apis.data.go.kr/B553881/newRegistlnfoService_02/getnewRegistlnfoService02?serviceKey=d0aea5c9cf7d8e653f3f0edc0f31ab09325738011a2e39c40225f3fada1b41d0&registYy=2025&registMt=12&vhctyAsortCode=1&sexdstn=%EB%82%A8%EC%9E%90&cnmCode=001236&agrde=3&hmmdImpSeNm=%EA%B5%AD%EC%82%B0', 'resultCode': '03', 'resultMsg': 'NODATA_ERROR', 'dtaCo': 0}


In [12]:
import os, requests, xml.etree.ElementTree as ET
from dotenv import load_dotenv

load_dotenv()
KEY = os.getenv("KEY")

URL = "https://apis.data.go.kr/B553881/newRegistlnfoService_02/getnewRegistlnfoService02"

def call(params):
    params = {k: v for k, v in params.items() if v is not None}
    r = requests.get(URL, params=params, timeout=20, headers={"User-Agent":"Mozilla/5.0"})
    txt = (r.text or "").strip()
    print("STATUS:", r.status_code)
    print("URL:", r.url.replace(KEY, "[KEY]"))
    print("HEAD:", txt[:120])

    if r.status_code != 200 or not txt.startswith("<"):
        return None

    root = ET.fromstring(txt)
    return {
        "resultCode": root.findtext(".//resultCode"),
        "resultMsg": root.findtext(".//resultMsg"),
        "dtaCo": root.findtext(".//dtaCo"),
    }

BASE = {
    "serviceKey": KEY,
    "registYy": "2025",
    "registMt": "12",
}

# 0) cnmCode 없이 (분모) — 이게 0이면 서비스/키/엔드포인트 문제
print("0) BASE only:", call(BASE))

# 1) cnmCode만 추가 — 여기서 no_data면 '코드체계 불일치' 가능성이 커짐
print("1) + cnmCode:", call({**BASE, "cnmCode": "001236"}))

# 2) 네가 쓰던 조건을 하나씩 추가
print("2) + vhctyAsortCode:", call({**BASE, "cnmCode":"001236", "vhctyAsortCode":"1"}))
print("3) + hmmdImpSeNm:", call({**BASE, "cnmCode":"001236", "vhctyAsortCode":"1", "hmmdImpSeNm":"국산"}))
print("4) + sexdstn:", call({**BASE, "cnmCode":"001236", "vhctyAsortCode":"1", "hmmdImpSeNm":"국산", "sexdstn":"남자"}))
print("5) + agrde:", call({**BASE, "cnmCode":"001236", "vhctyAsortCode":"1", "hmmdImpSeNm":"국산", "sexdstn":"남자", "agrde":"3"}))


STATUS: 200
URL: https://apis.data.go.kr/B553881/newRegistlnfoService_02/getnewRegistlnfoService02?serviceKey=[KEY]&registYy=2025&registMt=12
HEAD: <response><header><resultCode>00</resultCode><resultMsg>NORMAL_CODE</resultMsg></header><body><dtaCo>145607</dtaCo></bod
0) BASE only: {'resultCode': '00', 'resultMsg': 'NORMAL_CODE', 'dtaCo': '145607'}
STATUS: 200
URL: https://apis.data.go.kr/B553881/newRegistlnfoService_02/getnewRegistlnfoService02?serviceKey=[KEY]&registYy=2025&registMt=12&cnmCode=001236
HEAD: <response><header><resultCode>03</resultCode><resultMsg>NODATA_ERROR</resultMsg></header><body><dtaCo/></body></response
1) + cnmCode: {'resultCode': '03', 'resultMsg': 'NODATA_ERROR', 'dtaCo': ''}
STATUS: 200
URL: https://apis.data.go.kr/B553881/newRegistlnfoService_02/getnewRegistlnfoService02?serviceKey=[KEY]&registYy=2025&registMt=12&cnmCode=001236&vhctyAsortCode=1
HEAD: <response><header><resultCode>03</resultCode><resultMsg>NODATA_ERROR</resultMsg></header><body><dtaCo/></bod